# MAS v2 Baseline (No Trust Score)

Head → 3 Specialists → Final Reasoning. Benchmarks: CV-Bench, 3DSRBench. Samples: 10, 50, 100.

**Command-line alternative:** `bash run_h100.sh` or `python run_all.py`

In [ ]:
import sys
from pathlib import Path

# H100: sys.path.insert(0, "/home/jovyan/CY/Spatial_MAS")
ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "run_eval_mas_v2.py").exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from run_eval_mas_v2 import build_runners, run_experiment

In [ ]:
head_gen, spec_gen, reason_gen = build_runners(
    specialist_device="cuda",
    use_local_reasoning=True,
    reasoning_local_model_id="deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
)

In [ ]:
OUTPUT_BASE = ROOT / "results" / "mas_v2_baseline"
SAMPLE_SIZES = [10, 50, 100]
SEED = 42

for benchmark in ["cvbench", "3dsrbench"]:
    for n in SAMPLE_SIZES:
        print(f"\n>>> {benchmark} | {n} samples")
        out = run_experiment(
            benchmark=benchmark,
            head_generate=head_gen,
            specialist_generate=spec_gen,
            reasoning_generate=reason_gen,
            train_ratio=0.5,
            seed=SEED,
            output_dir=str(OUTPUT_BASE / benchmark / f"{n}samples"),
            max_samples=n,
        )
        print(f"  Train: {out['train_metrics']['accuracy']*100:.1f}% | Test: {out['test_metrics']['accuracy']*100:.1f}%")